# Award Topics: Create Input

Creates the `award_topics_input` table with awards that need BERT topic predictions. Selects awards with both `display_name` and `description` populated (strict eligibility, ~3.46M / 25% of all awards), excluding any already in `award_topics`.

Mirrors `topics_create_input.ipynb` for the awards pipeline (oxjob #123.1).

Processed rows are deleted from this table after each inference run, so subsequent runs pick up the remainder.

In [ ]:
-- Defensive: ensure openalex.awards.award_topics exists so the LEFT ANTI JOIN
-- below works on the very first run (oxjob #123.1). award_topics_merge_output
-- creates the same table at the end of the chain; CreateAwards also has a
-- matching bootstrap. Idempotent — no-op on subsequent runs.
CREATE TABLE IF NOT EXISTS openalex.awards.award_topics (
  award_id BIGINT,
  topics ARRAY<STRUCT<id: STRING, display_name: STRING, score: FLOAT, subfield: STRUCT<id: STRING, display_name: STRING>, field: STRUCT<id: STRING, display_name: STRING>, domain: STRUCT<id: STRING, display_name: STRING>>>,
  source STRING,
  created_datetime TIMESTAMP,
  updated_datetime TIMESTAMP
) USING DELTA;

In [ ]:
CREATE OR REPLACE TABLE openalex.awards.award_topics_input AS
SELECT
  a.id AS award_id,
  a.display_name,
  a.description
FROM openalex.awards.openalex_awards a
LEFT ANTI JOIN openalex.awards.award_topics at
  ON a.id = at.award_id
WHERE a.display_name IS NOT NULL
  AND a.description IS NOT NULL;

In [ ]:
OPTIMIZE openalex.awards.award_topics_input ZORDER BY (award_id);

In [ ]:
SELECT FORMAT_NUMBER(COUNT(*), 0) AS num_awards_to_process
FROM openalex.awards.award_topics_input;